In [4]:
import pandas as pd
import numpy as np

In [2]:
link = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [5]:
!curl -O {link}

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100 620.2k 100 620.2k   0      0  1.44M      0                              0
100 620.2k 100 620.2k   0      0  1.44M      0                              0
100 620.2k 100 620.2k   0      0  1.44M      0                              0


In [10]:
full_data_set = pd.read_csv('car_fuel_efficiency_2026.csv')

In [11]:
full_data_set.shape

(10000, 11)

In [12]:
full_data_set.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
#Q1
full_data_set.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [38]:
#data_set = full_data_set.copy()
data_set = full_data_set[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']].copy()

In [ ]:
#Question 2
data_set['horsepower'].median()

np.float64(254.0)

In [39]:
n = len(data_set)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = data_set.iloc[idx[:n_train]]
df_val = data_set.iloc[idx[n_train: n_train + n_val]]
df_test = data_set.iloc[idx[n_train + n_val:]]
    

In [40]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [41]:
y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

In [42]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [43]:
df_train.dtypes

engine_displacement      int64
horsepower             float64
vehicle_weight           int64
model_year               int64
dtype: object

In [72]:
hp_mean = df_train['horsepower'].mean()

base = [ 'engine_displacement', 'horsepower','vehicle_weight', 'model_year']
print(hp_mean)
df_train_median_hp  = df_train[base].fillna(hp_mean)
df_val_median_hp  = df_val[base].fillna(hp_mean)
df_test_median_hp  = df_test[base].fillna(hp_mean)

df_train_zero_hp  = df_train[base].fillna(0)
df_val_zero_hp  = df_val[base].fillna(0)
df_test_zero_hp  = df_test[base].fillna(0)


254.46338337605272


In [74]:
#print( type(df_test), type(df_test_median_hp))
df_test_zero_hp.isnull().sum()

engine_displacement    0
horsepower             0
vehicle_weight         0
model_year             0
dtype: int64

In [ ]:
#Did not use ofr Homework
def prepare_X(df):
    df = df.copy()
    
    #df['age'] = 2017 - df['year']
    features = base.copy()

    for v in [2, 3, 4]:
        df['num_doors_%d' % v] = (df.number_of_doors == v).astype(int)
        features.append('num_doors_%d' % v)

    for name, values in categorical.items():
        for value in values:
            df['%s_%s' % (name, value)] = (df[name] == value).astype(int)
            features.append('%s_%s' % (name, value))

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values

    return X

In [94]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [ ]:
def train_linear_regression_reg(X, y, r=0.00001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [102]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return round(np.sqrt(mse), 3)

In [ ]:
df_t

In [ ]:
#Q3
w0, w = train_linear_regression_reg(df_train_median_hp, y_train, r = 0.01 )

y_pred = w0 + df_val.dot(w)
print('Model with Mean HP: ', rmse(y_val, y_pred))

Model with Mean HP:  0.071


In [ ]:
#Q3
w0, w = train_linear_regression_reg(df_train_zero_hp, y_train, r = 0.01 )

y_pred = w0 + df_val.dot(w)
print('Model with 0 HP: ', rmse(y_val, y_pred))

Model with 0 HP:  0.071


In [ ]:
#Question 4
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    #X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(df_train_zero_hp, y_train, r=r)

    #X_val = prepare_X(df_val)
    y_pred = w0 + df_val_zero_hp.dot(w)
    score = rmse(y_val, y_pred)
    
    print(r, w0, score)

-0.01 -2.648449523810624 0.07149
0 -2.5488798807784008 0.07149
0.01 -2.4565256969286033 0.07149
0.1 -1.8524448492916248 0.07165
1 -0.5355295538031267 0.07274
5 -0.12874519343121463 0.07328
10 -0.06603998348980966 0.07337
100 -0.006760761336879879 0.07346


In [ ]:
## Question 5
rsme_scores = []
for s in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(data_set)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = data_set.iloc[idx[:n_train]]
    df_val = data_set.iloc[idx[n_train: n_train + n_val]]
    df_test = data_set.iloc[idx[n_train + n_val:]]

    df_train.reset_index(drop=True)
    df_val.reset_index(drop=True)
    df_test.reset_index(drop=True)

    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values

    del df_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']

    df_train = df_train.fillna(0)
    df_val = df_val.fillna(0)
    df_test = df_test.fillna(0)

    w0, w = train_linear_regression(df_train, y_train)

    y_pred = w0 + df_val.dot(w)
    score = rmse(y_val, y_pred)    

    rsme_scores.append(score)
    print(s, w0, score)

print (rsme_scores)
round(np.std(rsme_scores), 3)
    
    

0 -150.42885945053604 2.239
1 -151.61231559524666 2.202
2 -152.84068112584038 2.163
3 -152.33298096679223 2.204
4 -158.88856816457883 2.202
5 -151.8573533887713 2.246
6 -154.3717913727048 2.27
7 -155.0183525899584 2.191
8 -150.4381740048234 2.217
9 -154.36007488958072 2.207
[np.float64(2.239), np.float64(2.202), np.float64(2.163), np.float64(2.204), np.float64(2.202), np.float64(2.246), np.float64(2.27), np.float64(2.191), np.float64(2.217), np.float64(2.207)]


np.float64(0.029)

In [ ]:
#Question 6 - 

for s in [9]:
    n = len(data_set)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = data_set.iloc[idx[:n_train]]
    df_val = data_set.iloc[idx[n_train: n_train + n_val]]
    df_test = data_set.iloc[idx[n_train + n_val:]]

    #Concat df_train & df_val based on rows
    
    df_combined = pd.concat([df_train, df_val], axis = 0, ignore_index=True)
    
    #df_combined.reset_index(drop=True)
    df_test.reset_index(drop=True)

    #y_train = df_train.fuel_efficiency_mpg.values
    #y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values
    y_combined = df_combined.fuel_efficiency_mpg.values

    #del df_train['fuel_efficiency_mpg']
    #del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']
    del df_combined['fuel_efficiency_mpg']

    #df_train = df_train.fillna(0)
    #df_val = df_val.fillna(0)
    df_test = df_test.fillna(0)
    df_combined = df_combined.fillna(0)

    print('df comb: ', df_combined.shape)
    print('y comb: ', y_combined.shape)
    print('y test: ', y_test.shape)

    w0, w = train_linear_regression_reg(df_combined, y_combined, r=0.001)

    y_pred = w0 + df_test.dot(w)
    score = rmse(y_test, y_pred)    

    
    print(s, w0, score)



    
    

df comb:  (8000, 4)
y comb:  (8000,)
y test:  (2000,)
9 -152.37535772493302 2.236
